# SQLite, through a hostcall

The Lab has real SQLite — compiled to WebAssembly and vendored — and a
program reaches it the way it reaches the clock: by **asking its host**.

That distinction is the whole design and it is worth getting straight
before any SQL happens. There is no `sqlite` library inside the sandbox;
`notebooks/sql.ipynb` builds a small engine in Lua precisely because a
sealed program has no database in it. What exists here is a *connector*: a
program granted `host:sql/query` sends a message and rows come back. A
program not granted it cannot reach the database at all, which is what
makes the grant worth writing down.

This is also the production shape. `host/dhost_sql.c` answers the same two
calls over the system SQLite, so a program written here moves to the C
host unchanged.

In [ ]:
-- SQL is host-side, so this needs the swarm layer, and the `host` guest
-- library (v5.5.1_build7+) is what makes reaching it a line rather than a
-- page.
if type(swarm) ~= "table" then
  print("SKIP -- needs the Lab swarm runner (v5.5.1_build5 or newer)")
  df_sql_ready = false
elseif type(host) ~= "table" then
  print("SKIP -- needs the host library (v5.5.1_build7 or newer)")
  df_sql_ready = false
else
  df_sql_ready = true
  print("ready")
end

## Asking the host

A hostcall is a message, not a function call: `{tok, call, args}` out on
`host/calls`, `{tok, status, value}` back on `host/replies`, matched by a
**correlation token** because replies arrive in whatever order the host
answers them. By hand that is a page of plumbing in every program:

```lua
local calls   = queue.declare("host/calls",   { capacity = 8, exported = true })
local replies = queue.declare("host/replies", { capacity = 8 })
local tok = 0
local function hostcall(name, args)
  tok = tok + 1
  local mine = tok
  queue.push(calls, { tok = mine, call = name, args = args })
  while true do
    local _, reply = queue.wait({ replies })
    if reply.tok == mine then return reply end
  end
end
```

Since v5.5.1_build7 you do not write it. `host` is a guest library — a
*permanent*, baked into the runtime beside `queue` — that declares the
pair, correlates the tokens and checks the status, so a program says what
it wants instead of how to ask:

```lua
local db = host.sql.open("lab.db")
db.exec("INSERT INTO note (body) VALUES (?)", "hello")
local rows = db.query("SELECT body FROM note").rows
```

Nothing about the wire changed — the same three fields go out and the same
three come back, and `doc/Hostcall.md` is the protocol either way. What
changed is who writes the loop.

Two shapes, and the difference is the whole ergonomics of the thing:

- `db.exec` / `db.query` **raise** on a refusal, with the connector's own
  sentence in the message. That is the readable default.
- `db.try_exec` / `db.try_query` return `value, status, detail`. That is
  for a line that *expects* to be refused — every gate in this notebook.

The helper below runs a root program and hands back whatever it pushed to
its outbox, so the rest of this notebook is about SQL rather than
plumbing.

In [ ]:
-- Everything below shares this. The connector's configuration is in
-- `host/example.host.lua`'s shape on purpose: the deployment grants a
-- *scope* -- a directory -- and the program names its database inside it.
function run_sql(body, config)
  if not df_sql_ready then return {} end
  swarm.stop()
  swarm.start{
    root = [[
      local out = queue.lookup("outbox")
      function say(...) queue.push(out, table.concat({ ... }, " ")) end

      -- Which database is the program's business, and the config cannot
      -- make that choice for it. `open` is client-side: it binds the name
      -- into every call the handle makes and asks the host nothing yet.
      db = host.sql.open("lab.db")

      -- A result, reported. `query` returns the value directly (it raised
      -- if there was no value), so there is no status to unwrap here.
      function show(label, value)
        if type(value) == "table" and value.rows then
          say(label, "->", "ok,", tostring(#value.rows), "row(s)")
          for _, row in ipairs(value.rows) do
            local cs = {}
            for i, c in ipairs(row) do cs[i] = tostring(c) end
            say("   ", table.concat(cs, " | "))
          end
        else
          say(label, "->", "ok")
        end
      end

      -- A refusal this notebook set up on purpose, reported as the pass it
      -- is. Takes `try_*`'s three returns straight, so an unexpected
      -- *success* is what stands out -- a gate that quietly stopped
      -- holding used to look exactly like one that held.
      function refused(label, value, status, detail)
        if status == "ok" then
          say(label, "->", "UNEXPECTED -- this was supposed to be refused")
        else
          say(label, "->", "refused as designed (" .. tostring(status) .. "):",
              tostring(detail))
        end
      end
    ]] .. body,
    caps = { "queue:*", "host:sql/query", "host:sql/exec" },
    budget = { instructions = 20000000, memory_kb = 256 },
    max_instances = 4,
    connectors = { sql = config
      or { scope = "lab", access = "readwrite", max_result_rows = 64 } },
  }
  swarm.step(200)
  local out = swarm.drain("root", "outbox")
  -- Deliberately NOT stopped: the swarm, and the database it built,
  -- stay up so the Instances panel has something to show -- and
  -- something to download. The swarm.stop() above clears the previous
  -- run, so each cell still starts from an empty scope.
  return out
end

function show_sql(body, config)
  for _, line in ipairs(run_sql(body, config)) do print(tostring(line)) end
end
print("run_sql is ready")

## What the Lua engine could not do

`sql.ipynb` refuses joins, subqueries and grouping — honestly, because
writing a query planner is a different notebook. SQLite has all of them,
and the difference is not cosmetic: this is the same engine that will be
underneath in production.


In [ ]:
show_sql([[
  db.exec("CREATE TABLE account (id INTEGER PRIMARY KEY, name TEXT NOT NULL UNIQUE)")
  db.exec("CREATE TABLE hit (id INTEGER PRIMARY KEY, owner INTEGER NOT NULL, ms INTEGER)")
  for _, n in ipairs({ "ada", "annie", "grace" }) do
    db.exec("INSERT INTO account (name) VALUES (?)", n)
  end
  for _, h in ipairs({ {1,10}, {1,25}, {2,40}, {3,5}, {3,15}, {3,30} }) do
    db.exec("INSERT INTO hit (owner, ms) VALUES (?, ?)", h[1], h[2])
  end

  show("join + group by + order by", db.query([==[
    SELECT a.name, COUNT(h.id) AS n, SUM(h.ms) AS total
    FROM account a JOIN hit h ON h.owner = a.id
    GROUP BY a.name ORDER BY total DESC
  ]==]))

  show("a subquery", db.query(
    "SELECT name FROM account WHERE id IN (SELECT owner FROM hit WHERE ms > ?)", 30))

  show("a CTE", db.query(
    "WITH slow AS (SELECT owner FROM hit WHERE ms >= 25) SELECT COUNT(*) FROM slow"))
]])

## SQLite's own semantics, not an imitation of them

Constraints fire because SQLite enforces them, and `NULL` compares equal to
nothing — including itself. A hand-written engine has to remember to do
this; a real one cannot forget.


In [ ]:
show_sql([[
  db.exec("CREATE TABLE t (id INTEGER PRIMARY KEY, k TEXT NOT NULL UNIQUE, note TEXT)")
  db.exec("INSERT INTO t (k) VALUES ('one')")
  refused("a duplicate key",       db.try_exec("INSERT INTO t (k) VALUES ('one')"))
  refused("a NULL where NOT NULL", db.try_exec("INSERT INTO t (k) VALUES (NULL)"))
  db.exec("INSERT INTO t (k, note) VALUES ('two', NULL)")
  show("note = NULL",   db.query("SELECT id FROM t WHERE note = NULL"))
  show("note IS NULL",  db.query("SELECT id FROM t WHERE note IS NULL"))
]])

## The confinement, which is the weaker half

Here is the part to read before trusting this with anything.

The **contract** is the C host's exactly — same two calls, same shapes,
same read/write split. The **confinement** is not. `host/dhost_sql.c`
earns its confinement from three SQLite primitives no JavaScript driver
exposes: `sqlite3_set_authorizer`, `SQLITE_LIMIT_ATTACHED` and
`sqlite3_stmt_readonly`. Without them the escapes are gated on the
statement's *text*, which is a floor rather than a target.

So: build to the contract so your guest cannot tell the two apart, and do
not point this at a database that matters. Production is the C host.

Four gates, each visible below.


In [ ]:
show_sql([[
  db.exec("CREATE TABLE t (id INTEGER PRIMARY KEY, a TEXT)")

  -- 1. Transactions, ATTACH and PRAGMA are host state held against a
  -- guest, and the v1 encoding has nowhere to put a handle spanning calls.
  refused("BEGIN",   db.try_exec("BEGIN"))
  refused("PRAGMA",  db.try_exec("PRAGMA journal_mode = WAL"))
  refused("ATTACH",  db.try_exec("ATTACH DATABASE 'other.db' AS other"))

  -- 2. One statement per call. The drivers prepare the first and ignore
  -- the rest, so a second would ride in unauthorised AND unrun -- worse
  -- than either running it or refusing it. SQLite's own parser decides
  -- where a statement ends, so a `;` inside a literal is not a separator.
  refused("two statements", db.try_exec("INSERT INTO t (a) VALUES ('x'); DROP TABLE t"))
  show("a ; inside a literal", db.exec("INSERT INTO t (a) VALUES ('x;y')"))
  show("the table survived",   db.query("SELECT COUNT(*) FROM t"))

  -- 3. The parameter count must match exactly. Too few silently NULL-binds
  -- the rest, which is the same class of quiet wrongness as a truncated
  -- result.
  refused("too few params", db.try_exec("INSERT INTO t (id, a) VALUES (?, ?)", 99))

  -- 4. A name is a filename inside the granted scope, never a path. Escape
  -- is DENIED rather than clamped to something legal: a program that asked
  -- for the wrong database should hear so, not quietly get a different one.
  refused("a name that climbs out", host.try("sql/query",
    { db = "../secrets.db", sql = "SELECT 1" }))
]])

### The row cap refuses rather than truncating

A truncated result is a silent lie: the program gets rows, believes it got
*the* rows, and is wrong. So the cap is an error, and the guest pages with
`LIMIT`/`OFFSET` instead.

It is also checked while stepping a cursor rather than after materialising
— counting a hostile result set *after* building it in memory is a denial
of service the C host does not have.

In [ ]:
show_sql([[
  db.exec("CREATE TABLE n (i INTEGER PRIMARY KEY)")
  for i = 1, 12 do db.exec("INSERT INTO n (i) VALUES (?)", i) end
  refused("all 12, cap is 5", db.try_query("SELECT i FROM n"))
  show("paged to 5",         db.query("SELECT i FROM n LIMIT 5"))
]], { scope = "lab", access = "readwrite", max_result_rows = 5 })

### A grant that is read-only leaves `sql/exec` unwired

`access = "read"` is not a runtime check inside the connector — the write
call is simply never wired, so asking for it is `denied` the way any
ungranted call is. The capability and the configuration agree, which is
the property worth having.

The other half of a read grant is `create`, which follows it: a deployment
that cannot write cannot bring a database into existence either. So a
read-only scope with nothing in it yet has nothing to read, and says so
rather than handing back an empty database that looks like an empty table.
Both refusals are below.

In [ ]:
show_sql([[
  refused("a write",              db.try_exec("CREATE TABLE t (id INTEGER)"))
  refused("a read of what is not there", db.try_query("SELECT 1 AS one"))
]], { scope = "lab", access = "read", max_result_rows = 16 })

## The database is a file

The database lives in memory — a browser tab has no filesystem, so the
`scope` above names a directory rather than locating one, and what a
program builds inside it is gone when the kernel restarts.

It can still leave, and arrive. In the **Instances** panel:

- **Download .sqlite** exports it, one button per database the scope
  holds. What comes out begins `SQLite format 3`, because SQLite
  serialised it — `sqlite3`, a GUI, or another Lab session will open it.
- **Open .sqlite…**, beside the program picker, goes the other way. The
  file is staged under a name you can edit, and that name is how a program
  reaches it: a file staged as `orders.db` is the one `host.sql.open("orders.db")`
  finds. It opens when you press **Start**, because the scope is built
  when the swarm is. A file that is not a database is refused when you
  choose it, by name.

Those are buttons, so this notebook cannot press them — a cell drives the
kernel, not the page. What it *can* do is show that the bytes survive a
round trip, which is the property those buttons rest on: run the cell
below, then open the panel and download what it made.

In [ ]:
show_sql([[
  db.exec("CREATE TABLE note (id INTEGER PRIMARY KEY, body TEXT)")
  db.exec("INSERT INTO note (body) VALUES (?)", "written in a cell, readable in sqlite3")
  show("stored", db.query("SELECT body FROM note"))
  say("")
  say("Open the Instances panel and press Download .sqlite to take this away.")
]])

## Where this goes

`doc/Host.md`'s acceptance test is that **a guest must not be able to tell
two hosts apart**. For the contract that holds exactly, and now for the
engine too: a query that runs here runs on the C host, and a constraint
that fires here fires there. The `host` library is the same library on
both sides — it is baked into the runtime, not into either host — so the
code above is the code that ships.

What does not hold is the confinement, and the Lab says so rather than
letting you find out — in the panel, in the README, and in the refusals
above. Prototype the schema here; run it where the authorizer is.

Related notebooks: **A swarm, from a cell** for the layer this rides on,
**Messages and queues** for the queue API underneath every hostcall, and
**Building SQL in Lua** for what a program does when it has no host to
ask.